In [4]:
import pandas as pd

df = pd.read_csv('2019-Nov.csv')

df.head()

KeyboardInterrupt: 

In [ ]:
smartphone_df = df[df['category_code'] == 'electronics.smartphone'].copy()

In [ ]:
#정렬 + unknown처리
df = smartphone_df.sort_values(['user_session', 'event_time']).copy()
df['brand'] = df['brand'].fillna('unknown')

In [ ]:
grouped = df.groupby(['user_session', 'brand'])

#grouped = df.groupby(['user_session', 'user_id', 'product_id', 'brand'])

In [ ]:
result = []
for (session, brand), group in grouped:
    group = group.sort_values('event_time')  
    events = group['event_type'].tolist()

    
    # index 찾기
    view_idx = next((i for i, e in enumerate(events) if e == 'view'), None)
    cart_idx = next((i for i, e in enumerate(events) if e == 'cart'), None)
    purchase_idx = next((i for i, e in enumerate(events) if e == 'purchase'), None)
    
    has_view = view_idx is not None
    valid_view_cart = view_idx is not None and cart_idx is not None and view_idx < cart_idx
    valid_cart_purchase = cart_idx is not None and purchase_idx is not None and cart_idx < purchase_idx
    
    result.append([
        session, brand,
        has_view,
        valid_view_cart,
        valid_cart_purchase
    ])

funnel_df = pd.DataFrame(result, columns=[
    'user_session', 'brand',
    'view', 'view_to_cart', 'cart_to_purchase'
])

In [ ]:
import numpy as np
import pandas as pd

# 브랜드별 퍼널 집계
brand_funnel = (
    funnel_df.groupby('brand')
    .agg(
        view_sessions=('view', 'sum'),
        view_to_cart_sessions=('view_to_cart', 'sum'),
        cart_to_purchase_sessions=('cart_to_purchase', 'sum')
    )
    .reset_index()
)

# 전환율 계산
brand_funnel['view_to_cart_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['view_to_cart_sessions'] / brand_funnel['view_sessions'],
    0
)

brand_funnel['cart_to_purchase_rate'] = np.where(
    brand_funnel['view_to_cart_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_to_cart_sessions'],
    0
)

brand_funnel['total_conversion_rate'] = np.where(
    brand_funnel['view_sessions'] > 0,
    brand_funnel['cart_to_purchase_sessions'] / brand_funnel['view_sessions'],
    0
)

# 이탈률
brand_funnel['drop_view_to_cart'] = 1 - brand_funnel['view_to_cart_rate']
brand_funnel['drop_cart_to_purchase'] = 1 - brand_funnel['cart_to_purchase_rate']

# 표본 적은 브랜드 제거
brand_funnel = brand_funnel[brand_funnel['view_sessions'] >= 100].copy()

# 매출 데이터 결합
brand_revenue = (
    smartphone_df[smartphone_df['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean')
    )
    .reset_index()
)

brand_analysis = brand_funnel.merge(
    brand_revenue,
    on='brand',
    how='left'
).fillna(0)

# 보기 좋게 정렬
brand_analysis = brand_analysis.sort_values(
    by='total_conversion_rate',
    ascending=False
)

# 퍼센트 변환
rate_cols = [
    'view_to_cart_rate',
    'cart_to_purchase_rate',
    'total_conversion_rate',
    'drop_view_to_cart',
    'drop_cart_to_purchase'
]

brand_analysis[rate_cols] = brand_analysis[rate_cols] * 100

brand_analysis.head(20)

In [ ]:
import matplotlib.pyplot as plt

top10_conv = brand_analysis.sort_values(
    by='total_conversion_rate', ascending=False
).head(10)

plt.figure(figsize=(12, 6))
plt.bar(top10_conv['brand'], top10_conv['total_conversion_rate'])
plt.xticks(rotation=45)
plt.title('Top 10 Brands by Total Conversion Rate')
plt.xlabel('Brand')
plt.ylabel('Total Conversion Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
top10_view = brand_analysis.sort_values(
    by='view_sessions', ascending=False
).head(10)

plt.figure(figsize=(12, 6))
plt.bar(top10_view['brand'], top10_view['view_sessions'])
plt.xticks(rotation=45)
plt.title('Top 10 Brands by View Sessions')
plt.xlabel('Brand')
plt.ylabel('View Sessions')
plt.tight_layout()
plt.show()

In [ ]:
top10_revenue = brand_analysis.sort_values(
    by='revenue', ascending=False
).head(10)

plt.figure(figsize=(12, 6))
plt.bar(top10_revenue['brand'], top10_revenue['revenue'])
plt.xticks(rotation=45)
plt.title('Top 10 Brands by Revenue')
plt.xlabel('Brand')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 상위 브랜드만 (노이즈 제거)
plot_df = brand_analysis.sort_values(
    by='view_sessions', ascending=False
).head(20).copy()

plt.figure(figsize=(12, 8))

plt.scatter(
    plot_df['view_sessions'],                 # X축: 조회 수
    plot_df['total_conversion_rate'],        # Y축: 전환율
    s=plot_df['revenue'] / 1000,             # 점 크기: 매출 (스케일 조정)
    alpha=0.6
)

# 브랜드 이름 표시
for i, row in plot_df.iterrows():
    plt.text(
        row['view_sessions'],
        row['total_conversion_rate'],
        row['brand'],
        fontsize=9
    )

plt.xlabel('View Sessions')
plt.ylabel('Total Conversion Rate (%)')
plt.title('Brand Positioning: View vs Conversion (Size = Revenue)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
target_brands = ['apple', 'samsung', 'xiaomi']

compare_df = brand_analysis[
    brand_analysis['brand'].str.lower().isin(target_brands)
].copy()

compare_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 퍼널 단계
stages = ['View', 'View → Cart', 'Cart → Purchase']
x = np.arange(len(stages))
width = 0.25

plt.figure(figsize=(10,6))

for i, row in compare_df.iterrows():
    values = [
        row['view_sessions'],
        row['view_to_cart_sessions'],
        row['cart_to_purchase_sessions']
    ]
    
    plt.bar(x + i*width, values, width, label=row['brand'])

# x축 위치 가운데 정렬
plt.xticks(x + width, stages)

plt.ylabel('Number of Sessions')
plt.title('Funnel Count Comparison by Brand')
plt.legend()
plt.show()
for i, row in enumerate(compare_df.itertuples()):
    values = [
        row.view_sessions,
        row.view_to_cart_sessions,
        row.cart_to_purchase_sessions
    ]
    plt.bar(x + (i-1)*width, values, width, label=row.brand)

In [ ]:
target_brands = ['apple', 'samsung', 'xiaomi']

df_target = df[df['brand'].str.lower().isin(target_brands)].copy()

2

In [ ]:
#2 브랜드별 평균 / 중앙 판매가
brand_price_detail = (
    df_target[df_target['event_type'] == 'purchase']
    .groupby('brand')
    .agg(
        purchase_count=('price', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        min_price=('price', 'min'),
        max_price=('price', 'max')
    )
    .reset_index()
)

brand_price_detail

3

In [ ]:
#3. 상위 매출 product_id 비중
#몇 개 주력 모델에 몰려 있는지
product_revenue = (
    df_target[df_target['event_type'] == 'purchase']
    .groupby(['brand', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
)

product_revenue

In [ ]:
#상위 1개 / 3개 / 5개 상품 비중
def top_n_share(group, n):
    return group.sort_values('revenue', ascending=False).head(n)['revenue'].sum() / group['revenue'].sum()

top_share_rows = []

for brand, group in product_revenue.groupby('brand'):
    top_share_rows.append({
        'brand': brand,
        'top1_revenue_share': top_n_share(group, 1),
        'top3_revenue_share': top_n_share(group, 3),
        'top5_revenue_share': top_n_share(group, 5)
    })

top_share_df = pd.DataFrame(top_share_rows)
top_share_df

In [ ]:
top5_products = (
    product_revenue
    .sort_values(['brand', 'revenue'], ascending=[True, False])
    .groupby('brand')
    .head(5)
)

top5_products

4

In [ ]:
views_target = df_target[df_target['event_type'] == 'view'].copy()

session_view_depth = (
    views_target.groupby(['brand', 'user_session'])
    .agg(
        view_event_count=('product_id', 'size'),
        unique_view_products=('product_id', 'nunique')
    )
    .reset_index()
)

session_view_depth.head()

In [ ]:
brand_session_view_depth = (
    session_view_depth.groupby('brand')
    .agg(
        avg_view_events_per_session=('view_event_count', 'mean'),
        median_view_events_per_session=('view_event_count', 'median'),
        avg_unique_products_per_session=('unique_view_products', 'mean'),
        median_unique_products_per_session=('unique_view_products', 'median')
    )
    .reset_index()
)

brand_session_view_depth

5

In [ ]:
brand_session_compare = (
    views_target.groupby(['brand', 'user_session'])
    .agg(
        unique_models_compared=('product_id', 'nunique')
    )
    .reset_index()
)

brand_session_compare.head()

In [ ]:
brand_compare_depth = (
    brand_session_compare.groupby('brand')
    .agg(
        avg_model_compare_depth=('unique_models_compared', 'mean'),
        median_model_compare_depth=('unique_models_compared', 'median'),
        max_model_compare_depth=('unique_models_compared', 'max')
    )
    .reset_index()
)

brand_compare_depth

In [ ]:
compare_ratio = (
    brand_session_compare.groupby('brand')
    .apply(lambda g: pd.Series({
        'compare_2plus_session_ratio': (g['unique_models_compared'] >= 2).mean(),
        'compare_3plus_session_ratio': (g['unique_models_compared'] >= 3).mean()
    }))
    .reset_index()
)

compare_ratio

In [ ]:
brand_compare_depth = brand_compare_depth.merge(
    compare_ratio,
    on='brand',
    how='left'
)

brand_compare_depth

In [ ]:
final_compare = (
    compare_df
    .merge(
        brand_price_detail[['brand', 'median_price', 'min_price', 'max_price']],
        on='brand',
        how='left'
    )
    .merge(top_share_df, on='brand', how='left')
    .merge(brand_session_view_depth, on='brand', how='left')
    .merge(brand_compare_depth, on='brand', how='left')
)

final_compare

In [ ]:
pct_cols = [
    'top1_revenue_share',
    'top3_revenue_share',
    'top5_revenue_share',
    'compare_2plus_session_ratio',
    'compare_3plus_session_ratio'
]

final_compare[pct_cols] = final_compare[pct_cols] * 100

final_compare

시각화

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

brand_order = ['samsung', 'apple', 'xiaomi']

plot_df = final_compare.copy()
plot_df['brand'] = pd.Categorical(plot_df['brand'], categories=brand_order, ordered=True)
plot_df = plot_df.sort_values('brand')
plot_df

In [ ]:
stages = ['View', 'View → Cart', 'Cart → Purchase']
x = np.arange(len(stages))
width = 0.22

plt.figure(figsize=(10, 6))

for i, row in enumerate(plot_df.itertuples()):
    values = [
        row.view_sessions,
        row.view_to_cart_sessions,
        row.cart_to_purchase_sessions
    ]
    plt.bar(x + (i - 1) * width, values, width, label=row.brand)

plt.xticks(x, stages)
plt.ylabel('Number of Sessions')
plt.title('Brand Funnel Count Comparison')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
rate_cols = ['view_to_cart_rate', 'cart_to_purchase_rate', 'total_conversion_rate']
rate_labels = ['View → Cart', 'Cart → Purchase', 'Total']

x = np.arange(len(rate_labels))
width = 0.22

plt.figure(figsize=(10, 6))

for i, row in enumerate(plot_df.itertuples()):
    values = [row.view_to_cart_rate, row.cart_to_purchase_rate, row.total_conversion_rate]
    plt.bar(x + (i - 1) * width, values, width, label=row.brand)

plt.xticks(x, rate_labels)
plt.ylabel('Conversion Rate (%)')
plt.title('Funnel Conversion Rate by Brand')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, plot_df['avg_price'], width, label='Average Price')
plt.bar(x + width/2, plot_df['median_price'], width, label='Median Price')

plt.xticks(x, plot_df['brand'])
plt.ylabel('Price')
plt.title('Average vs Median Purchase Price by Brand')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(10, 6))
plt.bar(x - width, plot_df['top1_revenue_share'], width, label='Top 1')
plt.bar(x, plot_df['top3_revenue_share'], width, label='Top 3')
plt.bar(x + width, plot_df['top5_revenue_share'], width, label='Top 5')

plt.xticks(x, plot_df['brand'])
plt.ylabel('Revenue Share (%)')
plt.title('Top Product Revenue Concentration by Brand')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#세션 당 조회 상품 수
x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, plot_df['avg_unique_products_per_session'], width, label='Average')
plt.bar(x + width/2, plot_df['median_unique_products_per_session'], width, label='Median')

plt.xticks(x, plot_df['brand'])
plt.ylabel('Unique Products per Session')
plt.title('Viewed Product Depth per Session by Brand')
plt.legend()
plt.tight_layout()
plt.show()